# Pika-style Phone Video Generator (Wan2GP backend)

Run every cell top to bottom on a **GPU runtime** (Runtime > Change runtime type > GPU).

Free Colab GPU = 15GB T4 VRAM. Keep resolution at 480p and use the
`Wan 2.2 TextImage2Video 5B FastWan` model — other Wan2GP checkpoints will
likely run out of memory.


In [ ]:
#@title 1. Confirm GPU runtime
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU detected. Go to Runtime > Change runtime type > GPU, then re-run.")
print(result.stdout)


In [ ]:
#@title 2. (Optional) Mount Google Drive for persistent storage + final output
from google.colab import drive
drive.mount('/content/drive')

USE_GOOGLE_DRIVE_DATA = True  #@param {type:"boolean"}


In [ ]:
#@title 3. Clone Wan2GP
%cd /content
!git clone https://github.com/deepbeepmeep/Wan2GP.git
%cd /content/Wan2GP


In [ ]:
#@title 4. Install system + Python dependencies (Wan2GP + orchestrator)
!apt-get -qq install -y ffmpeg
!pip install -q -r /content/Wan2GP/requirements.txt

# Clone this repo's orchestrator code (edit the URL to your own fork/repo)
%cd /content
import os
REPO_URL = "https://github.com/<you>/<repo>.git"  #@param {type:"string"}
if not os.path.exists("/content/pika-video-generator"):
    !git clone {REPO_URL} /content/pika-video-generator

!pip install -q -r /content/pika-video-generator/requirements.txt


In [ ]:
#@title 5. Clone Wav2Lip + download pretrained checkpoint
%cd /content
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd /content/Wav2Lip
!mkdir -p checkpoints
# Original Google Drive links for wav2lip_gan.pth break often; this
# Hugging Face mirror is a commonly used stable alternative. If this URL
# is down for you, search "wav2lip_gan.pth huggingface" for a current
# mirror and drop it in checkpoints/wav2lip_gan.pth
!wget -q -O checkpoints/wav2lip_gan.pth \
  "https://huggingface.co/spaces/aiswaryaravindkumar/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth" \
  || echo "Download failed - manually place wav2lip_gan.pth in /content/Wav2Lip/checkpoints/"


In [ ]:
#@title 6. Launch Wan2GP headless (local only, no public link needed for this one)
import subprocess, time

%cd /content/Wan2GP
wan2gp_process = subprocess.Popen(
    ["python", "app.py", "--server-port", "7860", "--server-name", "127.0.0.1"],
)
print("Starting Wan2GP... waiting 30s for it to come up")
time.sleep(30)
print("Wan2GP should now be running on http://127.0.0.1:7860")


In [ ]:
#@title 7. Inspect Wan2GP's API (run once, check the printed endpoint name)
import sys
sys.path.insert(0, "/content/pika-video-generator/src")
from wan2gp_client import inspect_api
inspect_api()
# Compare the printed api_name / parameter order against
# WAN2GP_API_NAME and generate_clip() in wan2gp_client.py, and edit that
# file if they don't match your installed Wan2GP version.


In [ ]:
#@title 8. Launch your custom Pika-style UI (this is the link you open on your phone)
%cd /content/pika-video-generator/src
!python orchestrator_ui.py
